# Build Supplementary Output

Assembles a full 38,135-row supplementary CSV containing all matched scraped fields,
address standardization fields, and distance metrics that are not included in the
pruned final output.

**Input**:
- `data/1_derived/5_geocode_truck_stops/4_final_coordinates.csv` (intermediate with all columns)

**Output**:
- `output/1_derived/final_truck_stops_supplementary.csv`

In [ ]:
from pathlib import Path
import pandas as pd


def find_project_root(start=None, markers=('run_all.py', 'requirements.txt')):
    p = Path(start or Path.cwd()).resolve()
    for candidate in (p, *p.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise FileNotFoundError('Project root not found')


PROJECT_ROOT = find_project_root()
GEOCODE_DIR = PROJECT_ROOT / 'data' / '1_derived' / '5_geocode_truck_stops'
OUT_DIR = PROJECT_ROOT / 'output' / '1_derived'

IN_COORDS = GEOCODE_DIR / '4_final_coordinates.csv'
OUT_FILE = OUT_DIR / 'final_truck_stops_supplementary.csv'

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Define which supplementary columns to include from the intermediate file.
# Exclude working columns (row-ID lists, manual review, intermediate distance
# calcs) that are only used during pipeline processing.

all_cols = pd.read_csv(IN_COORDS, nrows=0).columns.tolist()

EXCLUDE_PATTERNS = [
    '_matches_row_ids',   # matching working columns
    '_unconstrained',     # intermediate distance variants
]
EXCLUDE_EXACT = {
    'all_matches',
    'Flagged', 'Flag_Reason', 'Is_Unclear_OCR',
    'max_distance_miles', 'max_distance_sources',
    'Mid_Lat', 'Mid_Long',
    'Manual_Match', 'Manual_Lat', 'Manual_Long',
}

supp_cols = []
for c in all_cols:
    if c in EXCLUDE_EXACT:
        continue
    if any(pat in c for pat in EXCLUDE_PATTERNS):
        continue
    supp_cols.append(c)

print(f'Intermediate has {len(all_cols)} columns')
print(f'Selecting {len(supp_cols)} supplementary columns (excluding {len(all_cols) - len(supp_cols)} working columns)')

In [ ]:
# Read selected columns from the intermediate file
print(f'Reading {len(supp_cols)} columns from {IN_COORDS.name} ...')
df = pd.read_csv(IN_COORDS, usecols=supp_cols, low_memory=False)
print(f'Loaded {len(df):,} rows x {len(df.columns)} columns')

In [ ]:
# Save
df.to_csv(OUT_FILE, index=False)
size_mb = OUT_FILE.stat().st_size / (1024 * 1024)
print(f'Saved: {OUT_FILE}')
print(f'Size: {size_mb:.1f} MB')
print(f'Shape: {df.shape}')